# Reading a GeoTIFF from the browser

`ipygeotiff` is a Jupyter - `geotiff.js` bridge. There are a couple of things to know to have it working:
- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.

In [ ]:
from urllib.parse import quote

from ipygeotiff import GeoTIFF

In [ ]:
geotiff = GeoTIFF()
geotiff  # this line is important, it creates the widget in the frontend

In [ ]:
url = "https://data.hydrosheds.org/file/hydrosheds-v2/DIR/1s/s10w050_DIR_1s_v2r0.tif"
proxy_url = (
    "https://my-proxy.david-brochart.workers.dev/?url="
    + quote(url, safe="")
)
tiff = await geotiff.from_url(proxy_url)

## Image bounds

The bounding box is expressed in the GeoTIFF's native coordinate reference system.

In [ ]:
image = await tiff.get_image()
bbox = image.bounding_box
bbox

## Read by pixel window

Image-level reads use pixel coordinates and return one shaped NumPy array per sample.

In [ ]:
pixel_rasters = await image.read_rasters(window=[0, 0, 4, 4])
pixel_rasters

## Read by geographic bounding box

GeoTIFF-level reads accept bounds in the image's native CRS. This example selects a small region near the upper-left corner and resamples it to 4 × 4 pixels.

In [ ]:
min_x, min_y, max_x, max_y = bbox
dx = (max_x - min_x) / 10_000
dy = (max_y - min_y) / 10_000
query_bbox = [min_x, max_y - dy, min_x + dx, max_y]

bbox_rasters = await tiff.read_rasters(
    bbox=query_bbox,
    width=4,
    height=4,
)
bbox_rasters